# 🗺️ NorthAir Intelligence — 04. Spatio-Temporal Analysis
## VIZ-A-THON 2026: Regional Pollution Dynamics Across North India (2025)

### Objectives:
1. Compare particulate pollution across Chandigarh, Punjab, Haryana, and Delhi.
2. Evaluate city-level and station-level concentration rankings.
3. Quantify monthly and seasonal variations:
   - **Winter** (Dec, Jan, Feb): Deep atmospheric inversion & stagnant dispersion.
   - **Pre-Monsoon / Summer** (Mar, Apr, May): High temperatures & mineral dust transport.
   - **Monsoon** (Jun, Jul, Aug, Sep): Intense precipitation & wet scavenging.
   - **Post-Monsoon** (Oct, Nov): Crop residue burning episodes & dropping boundary layer.
4. Calculate comprehensive parametric and non-parametric distributions.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 35)

# Load analytical datasets
analytics_dir = Path("../data/processed/analytics")
monthly_df = pd.read_parquet(analytics_dir / "monthly_pollution.parquet")
state_df = pd.read_parquet(analytics_dir / "state_comparison.parquet")
city_df = pd.read_parquet(analytics_dir / "city_comparison.parquet")
station_df = pd.read_parquet(analytics_dir / "station_rankings.parquet")

print("Analytics datasets loaded successfully.")


### 1. State-Wise Particulate Matter Comparison
Let us examine the distribution of PM2.5 and PM10 concentrations across the four states and union territories.
The official 24-hour CPCB National Ambient Air Quality Standard (NAAQS) is **60 µg/m³ for PM2.5** and **100 µg/m³ for PM10**.


In [ ]:
display(state_df[['State', 'Station_Count', 'PM25_Mean', 'PM25_Median', 'PM25_P90', 'PM25_Exceed_60_Pct', 'PM10_Mean', 'PM10_Median', 'PM10_Exceed_100_Pct']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# State Mean PM2.5
sns.barplot(data=state_df, x='State', y='PM25_Mean', palette='Reds_r', ax=axes[0])
axes[0].set_title("Annual Mean PM2.5 by State / UT (2025)", fontweight='bold')
axes[0].set_ylabel("Mean PM2.5 (µg/m³)")
axes[0].axhline(60, color='black', linestyle='--', label='CPCB 24h Standard (60 µg/m³)')
axes[0].legend()
for p in axes[0].patches:
    axes[0].annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')

# State Mean PM10
sns.barplot(data=state_df, x='State', y='PM10_Mean', palette='Oranges_r', ax=axes[1])
axes[1].set_title("Annual Mean PM10 by State / UT (2025)", fontweight='bold')
axes[1].set_ylabel("Mean PM10 (µg/m³)")
axes[1].axhline(100, color='black', linestyle='--', label='CPCB 24h Standard (100 µg/m³)')
axes[1].legend()
for p in axes[1].patches:
    axes[1].annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=10, xytext=(0, 2), textcoords='offset points')

plt.tight_layout()
plt.show()


### 2. Monthly Pollution Evolution Across States
Pollution exhibits dramatic seasonal swings in North India, dropping during the wet summer monsoon and surging in post-monsoon / winter.


In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_df, x='Month', y='PM25_Mean', hue='State', marker='o', linewidth=2.5)
plt.title("Monthly Mean PM2.5 Concentration Trajectory by State (2025)", fontsize=13, fontweight='bold')
plt.xlabel("Month of Year (1=Jan to 12=Dec)")
plt.ylabel("Mean PM2.5 (µg/m³)")
plt.xticks(range(1, 13), ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
plt.axhline(60, color='gray', linestyle=':', label='National Standard (60 µg/m³)')
plt.legend(title="State / UT")
plt.tight_layout()
plt.show()


### 3. Top Polluted Urban Centers (City Rankings)
Ranking North Indian cities by annual average PM2.5 exposure.


In [ ]:
top_cities = city_df.head(15)
display(top_cities[['City_Rank_PM25', 'City', 'State', 'PM25_Mean', 'PM25_Median', 'PM25_P90', 'PM25_Exceed_60_Pct', 'PM10_Mean']])

plt.figure(figsize=(10, 6))
sns.barplot(data=top_cities, x='PM25_Mean', y='City', hue='State', dodge=False, palette='viridis')
plt.title("Top 15 Most Polluted Cities Ranked by Mean PM2.5 (2025)", fontsize=13, fontweight='bold')
plt.xlabel("Annual Mean PM2.5 (µg/m³)")
plt.axvline(60, color='red', linestyle='--', label='CPCB 24h Standard (60 µg/m³)')
plt.legend(title="State")
plt.tight_layout()
plt.show()


### 4. Summary of Spatio-Temporal Discoveries
1. **Seasonal Monsoonal Washing**: Across all four states, July and August represent the cleanest months ($PM_{2.5} < 35 \, \mu\text{g/m}^3$) due to precipitation washout.
2. **Post-Monsoon Winter Shock**: November experiences the peak network-wide particulate surge ($>200 \, \mu\text{g/m}^3$ state averages in Delhi and Haryana).
3. **Spatial Hierarchy**: Delhi and Haryana consistently report higher annual mean and extreme PM2.5 than Punjab and Chandigarh.
